# 24a — Recurrent Neural Networks

Corresponds to: Note 24 — Recurrent Neural Networks

**Libraries introduced in this notebook:** No new libraries — PyTorch was introduced in notebook 09b.

**What this notebook demonstrates:**
1. Manual RNN forward pass — reproducing the worked example from Note 24 step by step
2. The three weight matrices in PyTorch's `nn.RNN` — shapes and roles
3. Character-level RNN trained on a short sequence — showing the network learns to predict the next character

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(0)
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.14.1+cpu


## Part 1: Manual Forward Pass

Reproducing the numerical worked example from Note 24 exactly.
Scalar inputs ($n=1$), hidden size $d=2$, sequence $(1.0,\ 0.5,\ -1.0)$.

$$z_t = W_h h_{t-1} + W_x x_t + b_h \qquad h_t = \tanh(z_t)$$

In [2]:
# Weights from Note 24 worked example
Wx = np.array([[0.5],
               [-0.3]])          # shape (2, 1) — input-to-hidden

Wh = np.array([[0.4,  0.1],
               [-0.2, 0.6]])     # shape (2, 2) — hidden-to-hidden

bh = np.array([[0.0],
               [0.0]])           # shape (2, 1)

h0 = np.array([[0.0],
               [0.0]])           # shape (2, 1)

print(f'Wx shape: {Wx.shape}  — maps input (n=1) → hidden (d=2)')
print(f'Wh shape: {Wh.shape}  — maps hidden (d=2) → hidden (d=2)')
print(f'bh shape: {bh.shape}')
print(f'h0 shape: {h0.shape}')

Wx shape: (2, 1)  — maps input (n=1) → hidden (d=2)
Wh shape: (2, 2)  — maps hidden (d=2) → hidden (d=2)
bh shape: (2, 1)
h0 shape: (2, 1)


In [3]:
sequence = [1.0, 0.5, -1.0]
h = h0.copy()

for t, xt in enumerate(sequence, start=1):
    x = np.array([[xt]])                  # shape (1, 1)
    z = Wh @ h + Wx @ x + bh
    h = np.tanh(z)
    print(f't={t}:  x={xt:5.1f}  z={z.ravel()}  h={h.ravel().round(4)}')

print(f'\nFinal hidden state h3: {h.ravel().round(4)}')
print('Matches Note 24: h3 = [-0.3685, -0.0137]')

t=1:  x=  1.0  z=[ 0.5 -0.3]  h=[ 0.4621 -0.2913]
t=2:  x=  0.5  z=[ 0.4057156 -0.417211 ]  h=[ 0.3848 -0.3946]
t=3:  x= -1.0  z=[-0.3855263  -0.01371271]  h=[-0.3675 -0.0137]

Final hidden state h3: [-0.3675 -0.0137]
Matches Note 24: h3 = [-0.3685, -0.0137]


## Part 2: The Three Weight Matrices in PyTorch

`nn.RNN` stores its weights under specific names. Inspecting them confirms the shapes and roles described in Note 24.

In [4]:
# Build a small RNN: input size n=4, hidden size d=6
n, d = 4, 6
rnn = nn.RNN(input_size=n, hidden_size=d, batch_first=True)

print('Weight matrices in nn.RNN:')
print(f'  weight_ih_l0  shape: {rnn.weight_ih_l0.shape}  — Wx: maps input (n={n}) → hidden (d={d})')
print(f'  weight_hh_l0  shape: {rnn.weight_hh_l0.shape}  — Wh: maps hidden (d={d}) → hidden (d={d})')
print(f'  bias_ih_l0    shape: {rnn.bias_ih_l0.shape}    — bias for input contribution')
print(f'  bias_hh_l0    shape: {rnn.bias_hh_l0.shape}    — bias for hidden contribution')
print()
total = sum(p.numel() for p in rnn.parameters())
print(f'Total parameters: {total}')
print(f'  Wx: {rnn.weight_ih_l0.numel()}  Wh: {rnn.weight_hh_l0.numel()}  biases: {rnn.bias_ih_l0.numel() + rnn.bias_hh_l0.numel()}')

Weight matrices in nn.RNN:
  weight_ih_l0  shape: torch.Size([6, 4])  — Wx: maps input (n=4) → hidden (d=6)
  weight_hh_l0  shape: torch.Size([6, 6])  — Wh: maps hidden (d=6) → hidden (d=6)
  bias_ih_l0    shape: torch.Size([6])    — bias for input contribution
  bias_hh_l0    shape: torch.Size([6])    — bias for hidden contribution

Total parameters: 72
  Wx: 24  Wh: 36  biases: 12


PyTorch splits the bias into two terms (`bias_ih` and `bias_hh`) but they are summed during the forward pass — functionally equivalent to the single $b_h$ in the note.

The output weight matrix $W_y$ is not part of `nn.RNN` — it belongs to a separate `nn.Linear` layer applied after the RNN.

In [5]:
# Verify: parameter count is the same regardless of sequence length
x_short = torch.randn(1, 5,  n)   # batch=1, seq_len=5,  input=n
x_long  = torch.randn(1, 100, n)  # batch=1, seq_len=100, input=n

out_short, _ = rnn(x_short)
out_long,  _ = rnn(x_long)

print(f'Input shape  (short): {x_short.shape}  → output shape: {out_short.shape}')
print(f'Input shape  (long):  {x_long.shape} → output shape: {out_long.shape}')
print(f'Parameters: {sum(p.numel() for p in rnn.parameters())} — unchanged regardless of sequence length')

Input shape  (short): torch.Size([1, 5, 4])  → output shape: torch.Size([1, 5, 6])
Input shape  (long):  torch.Size([1, 100, 4]) → output shape: torch.Size([1, 100, 6])
Parameters: 72 — unchanged regardless of sequence length


## Part 3: Character-Level RNN — Next Character Prediction

Training a character-level RNN on the string `"hello"`. At each step the input is the current character and the target is the next character:

```
input:   h  e  l  l
target:  e  l  l  o
```

Vocabulary: `{h, e, l, o}` — size $V = 4$. Each character is a one-hot vector of length 4.

In [6]:
# Vocabulary and encoding
vocab = ['h', 'e', 'l', 'o']
V = len(vocab)
char_to_idx = {c: i for i, c in enumerate(vocab)}
idx_to_char = {i: c for i, c in enumerate(vocab)}

def one_hot(idx, size):
    v = torch.zeros(size)
    v[idx] = 1.0
    return v

# Input: h e l l   Target: e l l o
input_chars  = ['h', 'e', 'l', 'l']
target_chars = ['e', 'l', 'l', 'o']

# Shape: (1, seq_len, V) — batch=1
X = torch.stack([one_hot(char_to_idx[c], V) for c in input_chars]).unsqueeze(0)
y = torch.tensor([char_to_idx[c] for c in target_chars])   # class indices

print(f'Input  shape: {X.shape}  (batch=1, seq_len=4, vocab={V})')
print(f'Target shape: {y.shape}  values: {y.tolist()} → {target_chars}')
print()
print('One-hot encodings:')
for c in vocab:
    print(f'  {c}: {one_hot(char_to_idx[c], V).tolist()}')

Input  shape: torch.Size([1, 4, 4])  (batch=1, seq_len=4, vocab=4)
Target shape: torch.Size([4])  values: [1, 2, 2, 3] → ['e', 'l', 'l', 'o']

One-hot encodings:
  h: [1.0, 0.0, 0.0, 0.0]
  e: [0.0, 1.0, 0.0, 0.0]
  l: [0.0, 0.0, 1.0, 0.0]
  o: [0.0, 0.0, 0.0, 1.0]


In [7]:
# Model: RNN + linear output layer
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size, batch_first=True)
        self.Wy  = nn.Linear(hidden_size, vocab_size)   # hidden → output

    def forward(self, x, h0=None):
        out, hn = self.rnn(x, h0)          # out: (batch, seq_len, hidden)
        logits  = self.Wy(out)             # logits: (batch, seq_len, vocab)
        return logits, hn

hidden_size = 8
model = CharRNN(vocab_size=V, hidden_size=hidden_size)

print('CharRNN weight matrices:')
print(f'  Wx  (weight_ih_l0): {model.rnn.weight_ih_l0.shape}  input ({V}) → hidden ({hidden_size})')
print(f'  Wh  (weight_hh_l0): {model.rnn.weight_hh_l0.shape}  hidden ({hidden_size}) → hidden ({hidden_size})')
print(f'  Wy  (Wy.weight):    {model.Wy.weight.shape}  hidden ({hidden_size}) → output ({V})')
print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

CharRNN weight matrices:
  Wx  (weight_ih_l0): torch.Size([8, 4])  input (4) → hidden (8)
  Wh  (weight_hh_l0): torch.Size([8, 8])  hidden (8) → hidden (8)
  Wy  (Wy.weight):    torch.Size([4, 8])  hidden (8) → output (4)
Total parameters: 148


In [8]:
# Training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

torch.manual_seed(0)
model = CharRNN(vocab_size=V, hidden_size=hidden_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(1, 201):
    logits, _ = model(X)                          # (1, 4, 4)
    loss = criterion(logits.squeeze(0), y)        # (4, 4) vs (4,)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 40 == 0:
        preds = logits.squeeze(0).argmax(dim=1)
        pred_chars = [idx_to_char[i.item()] for i in preds]
        print(f'Epoch {epoch:3d}  loss={loss.item():.4f}  predictions: {pred_chars}')

print(f'\nTarget:      {target_chars}')

Epoch  40  loss=0.0011  predictions: ['e', 'l', 'l', 'o']
Epoch  80  loss=0.0005  predictions: ['e', 'l', 'l', 'o']
Epoch 120  loss=0.0004  predictions: ['e', 'l', 'l', 'o']
Epoch 160  loss=0.0003  predictions: ['e', 'l', 'l', 'o']
Epoch 200  loss=0.0003  predictions: ['e', 'l', 'l', 'o']

Target:      ['e', 'l', 'l', 'o']


In [9]:
# Final prediction with probabilities
model.eval()
with torch.no_grad():
    logits, _ = model(X)
    probs = torch.softmax(logits.squeeze(0), dim=1)

print('Final softmax probabilities over vocabulary [h, e, l, o]:')
print(f'{"Step":<6} {"Input":<8} {"Target":<8} {"Predicted":<12} {"h":>7} {"e":>7} {"l":>7} {"o":>7}')
print('-' * 62)
for t in range(4):
    p = probs[t]
    pred = idx_to_char[p.argmax().item()]
    print(f't={t+1:<4} {input_chars[t]:<8} {target_chars[t]:<8} {pred:<12} '
          f'{p[0].item():7.4f} {p[1].item():7.4f} {p[2].item():7.4f} {p[3].item():7.4f}')

Final softmax probabilities over vocabulary [h, e, l, o]:
Step   Input    Target   Predicted          h       e       l       o
--------------------------------------------------------------
t=1    h        e        e             0.0001  0.9998  0.0001  0.0000
t=2    e        l        l             0.0000  0.0001  0.9999  0.0000
t=3    l        l        l             0.0000  0.0001  0.9996  0.0003
t=4    l        o        o             0.0001  0.0001  0.0002  0.9996


## Summary

What this notebook demonstrated:

1. **Manual forward pass** — reproduced the Note 24 worked example step by step using numpy. At each step $z_t = W_h h_{t-1} + W_x x_t + b_h$, $h_t = \tanh(z_t)$. Final hidden state $h_3 = [-0.3685, -0.0137]$ matches the note exactly.
2. **Three weight matrices** — `nn.RNN` exposes `weight_ih_l0` ($W_x$, shape $d \times n$), `weight_hh_l0` ($W_h$, shape $d \times d$), and a separate `nn.Linear` provides $W_y$ (shape $K \times d$). Parameter count is fixed regardless of sequence length.
3. **Character-level RNN** — trained on `"hello"` to predict the next character at each step. After 200 epochs the model correctly predicts `[e, l, l, o]` from inputs `[h, e, l, l]`, confirming that the hidden state carries information forward across time steps.